# TVB + Basal Ganglia Izhikevich SNN (NEST) Developer Walkthrough

Language: [English](./tvb_multiscale_basal_ganglia_nest_developer_walkthrough.ipynb) | [Chinese](./tvb_multiscale_basal_ganglia_nest_developer_walkthrough_zh.ipynb)

This notebook shows how to build the **basal ganglia spiking network + TVB whole-brain co-simulation** step by step.

It is intentionally written for developers:

- we do **not** hide the workflow behind `main_example()`;
- we build the TVB simulator, the NEST BG network, and the TVB<->NEST interface explicitly;
- every build step is annotated so you can see **where to change parameters, topology, devices, interface behavior, and quick-look analysis**.

The implementation demonstrated here comes from the `tvb_multiscale` code under the repository `src/` tree.
The notebook supports both layouts used in this project history:

- `src/tvb-multiscale`
- `src/tvb_multiscale`


## Build Pipeline

The co-simulation is assembled in five stages:

1. Load and prepare the **BG + cortex structural connectivity** used by TVB.
2. Build a **TVB `CoSimulator`** with a `ReducedWongWangExcIO` whole-brain model.
3. Build the **basal ganglia spiking network** in NEST with `BasalGangliaIzhikevichBuilder`.
4. Build the **TVB<->NEST interface** with `RedWWexcIOBuilder` specialized for the BG model.
5. Optionally configure the simulator and run a short co-simulation.

This notebook focuses on the **NEST** path because it is the most explicit end-to-end implementation for the BG model in this branch.

In [ ]:
from pathlib import Path
import os
import sys


def find_virtualbrain_root(start: Path) -> Path:
    """Find the project root that contains the notebook folder and a TVB multiscale source tree."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if not (candidate / "notebook").exists():
            continue
        if (candidate / "src" / "tvb-multiscale").exists():
            return candidate
        if (candidate / "src" / "tvb_multiscale").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the virtualbrain project root from the current working directory."
    )


def find_tvb_multiscale_roots(project_root: Path) -> tuple[Path, Path]:
    """Return (repo_root, import_root) for either repository layout used in this project."""
    hyphen_repo_root = project_root / "src" / "tvb-multiscale"
    if hyphen_repo_root.exists():
        return hyphen_repo_root, hyphen_repo_root

    underscore_pkg_root = project_root / "src" / "tvb_multiscale"
    if underscore_pkg_root.exists():
        return underscore_pkg_root, project_root / "src"

    raise FileNotFoundError(
        "Could not locate a tvb_multiscale source tree under project_root/src."
    )


print("Locating the virtualbrain project root...")
print(Path.cwd())
PROJECT_ROOT = find_virtualbrain_root(Path.cwd())
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
TVB_MULTISCALE_ROOT, TVB_MULTISCALE_IMPORT_ROOT = find_tvb_multiscale_roots(PROJECT_ROOT)

extra_sys_paths = [TVB_MULTISCALE_IMPORT_ROOT]
for path in extra_sys_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"PROJECT_ROOT            = {PROJECT_ROOT}")
print(f"NOTEBOOK_DIR            = {NOTEBOOK_DIR}")
print(f"TVB_MULTISCALE_ROOT     = {TVB_MULTISCALE_ROOT}")
print(f"TVB_MULTISCALE_IMPORT   = {TVB_MULTISCALE_IMPORT_ROOT}")


## NEST Environment Prerequisites

The NEST-specific config inside `tvb_multiscale.tvb_nest` is evaluated at import time.
That means the following environment variables must already exist **before** importing the NEST builders:

- `NEST_INSTALL_DIR`
- `NEST_PYTHON_PREFIX`

If you normally start Jupyter from a shell where NEST is already configured, the next cell should pass.
If not, define the variables there first, then rerun the import cells.

In [ ]:
# Example manual setup if you need it:
# os.environ["NEST_INSTALL_DIR"] = r"/path/to/nest/install"
# os.environ["NEST_PYTHON_PREFIX"] = r"/path/to/nest/python"


def configure_default_nest_env(project_root: Path, multiscale_root: Path) -> None:
    """Populate the minimal NEST env vars from a local user install if available."""
    if not (os.environ.get("NEST_INSTALL_DIR") and os.environ.get("NEST_PYTHON_PREFIX")):
        local_nest_root = project_root / ".local-opt" / "nest"
        versioned_candidates = sorted(local_nest_root.glob("nest-*"), reverse=True) if local_nest_root.exists() else []
        py_version = f"python{sys.version_info.major}.{sys.version_info.minor}"

        for candidate in versioned_candidates:
            python_prefix = candidate / "lib" / py_version / "site-packages"
            if (candidate / "bin" / "nest-config").exists() and python_prefix.exists():
                os.environ.setdefault("NEST_INSTALL_DIR", str(candidate))
                os.environ.setdefault("NEST_PYTHON_PREFIX", str(python_prefix))
                os.environ.setdefault("NEST_DATA_DIR", str(candidate / "share" / "nest"))
                os.environ.setdefault("NEST_MODULE_PATH", str(candidate / "lib" / "nest"))
                os.environ.setdefault("SLI_PATH", str(candidate / "share" / "nest" / "sli"))
                break

    os.environ.setdefault("MYMODULES_DIR", str(multiscale_root / "tvb_nest" / "nest" / "modules"))
    os.environ.setdefault("MYMODULES_BLD_DIR", str(project_root / ".nest_modules_builds"))
    Path(os.environ["MYMODULES_BLD_DIR"]).mkdir(parents=True, exist_ok=True)


configure_default_nest_env(PROJECT_ROOT, TVB_MULTISCALE_ROOT)

required_env = ["NEST_INSTALL_DIR", "NEST_PYTHON_PREFIX"]
missing_env = [name for name in required_env if not os.environ.get(name)]

if missing_env:
    raise EnvironmentError(
        "Missing required environment variables for the NEST-based walkthrough: "
        + ", ".join(missing_env)
        + "\nSet them in the shell before starting Jupyter, or define them in this cell before importing tvb_multiscale.tvb_nest."
    )

if os.environ["NEST_PYTHON_PREFIX"] not in sys.path:
    sys.path.insert(0, os.environ["NEST_PYTHON_PREFIX"])

print("NEST environment looks configured.")
for name in required_env + ["MYMODULES_DIR", "MYMODULES_BLD_DIR"]:
    print(f"{name} = {os.environ[name]}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint
from IPython.display import display

# Compatibility aliases for older TVB branches running on modern NumPy.
for alias, value in {
    "float": float,
    "int": int,
    "complex": complex,
    "bool": bool,
    "object": object,
    "long": int,
}.items():
    if not hasattr(np, alias):
        setattr(np, alias, value)

from tvb.datatypes.connectivity import Connectivity
try:
    from tvb.simulator.models.reduced_wong_wang_exc_io import ReducedWongWangExcIO
except ModuleNotFoundError:
    from tvb.contrib.scripts.models.reduced_wong_wang_exc_io import ReducedWongWangExcIO

from tvb_multiscale.core.tvb.simulator_builder import SimulatorBuilder
from tvb_multiscale.tvb_nest.config import Config
from tvb_multiscale.tvb_nest.nest_models.builders.models.basal_ganglia_izhikevich import (
    BasalGangliaIzhikevichBuilder,
)
from tvb_multiscale.tvb_nest.interfaces.builders.models.red_ww_basal_ganglia_izhikevich import (
    RedWWexcIOBuilder as BasalGangliaRedWWexcIOBuilder,
)


## Load the BG + Cortex Connectivity Used by TVB

The repository ships a BG-extended structural connectome under:

- `examples/data/basal_ganglia_conn/conn_denis_weights.txt`
- `examples/data/basal_ganglia_conn/BGplusAAL_tract_lengths.txt`
- `examples/data/basal_ganglia_conn/aal_plus_BG_centers.txt`

Important developer note:

- the current dataset already has the first 10 region labels arranged as:
  `GPe, GPi, STN, Striatum, Thalamus` (left/right);
- the BG NEST builder assumes exactly this node-id layout (`0..9`);
- therefore this walkthrough **keeps all 10 BG/thalamic nodes intact**.

Some historical example scripts in the repository remove thalamus nodes before building the simulation.
With the current data files, that would break the node-id convention expected by `BasalGangliaIzhikevichBuilder`.

In [ ]:
def find_bg_data_path(project_root: Path, multiscale_root: Path) -> Path:
    candidates = [
        multiscale_root / "examples" / "data" / "basal_ganglia_conn",
        project_root / "notebook" / "data" / "basal_ganglia_conn",
        project_root / "data" / "basal_ganglia_conn",
    ]
    required_files = {
        "conn_denis_weights.txt",
        "BGplusAAL_tract_lengths.txt",
        "aal_plus_BG_centers.txt",
    }
    for candidate in candidates:
        if candidate.exists() and required_files.issubset({path.name for path in candidate.iterdir()}):
            return candidate
    raise FileNotFoundError(
        "Could not locate the basal_ganglia_conn dataset. Checked: "
        + ", ".join(str(candidate) for candidate in candidates)
    )


DATA_PATH = find_bg_data_path(PROJECT_ROOT, TVB_MULTISCALE_ROOT)
print(f"DATA_PATH = {DATA_PATH}")

weights = np.loadtxt(DATA_PATH / "conn_denis_weights.txt")
centres = np.loadtxt(DATA_PATH / "aal_plus_BG_centers.txt", usecols=range(1, 4))
region_labels = np.loadtxt(DATA_PATH / "aal_plus_BG_centers.txt", dtype="str", usecols=(0,))
tract_lengths = np.loadtxt(DATA_PATH / "BGplusAAL_tract_lengths.txt")

expected_bg_order = [
    "GPe_Left", "GPe_Right",
    "GPi_Left", "GPi_Right",
    "STN_Left", "STN_Right",
    "Striatum_Left", "Striatum_Right",
    "Thal_Left", "Thal_Right",
]
actual_bg_order = region_labels[:10].tolist()
assert actual_bg_order == expected_bg_order, (
    "This walkthrough assumes that the first 10 regions match the BG node layout expected by "
    "BasalGangliaIzhikevichBuilder.\n"
    f"Expected: {expected_bg_order}\n"
    f"Actual:   {actual_bg_order}"
)

number_of_regions = len(region_labels)
speed = 4.0
min_tract_length = speed * 0.1

# Historical model-specific pruning used in the branch example.
# We keep the 10 BG/thalamic nodes, but remove selected BG<->cortex projections.
bg_to_cortex_nodes = [0, 1, 2, 3, 6, 7]   # GPe, GPi, Striatum
cortex_nodes = slice(10, number_of_regions)
weights[bg_to_cortex_nodes, cortex_nodes] = 0.0
tract_lengths[bg_to_cortex_nodes, cortex_nodes] = min_tract_length

cortex_to_gpe_gpi_nodes = [0, 1, 2, 3]    # GPe, GPi
weights[cortex_nodes, cortex_to_gpe_gpi_nodes] = 0.0
tract_lengths[cortex_nodes, cortex_to_gpe_gpi_nodes] = min_tract_length

connectivity = Connectivity(
    region_labels=region_labels,
    weights=weights,
    centres=centres,
    tract_lengths=tract_lengths,
)
connectivity.configure()

bg_node_ids = list(range(10))
bg_mapping = {node_id: connectivity.region_labels[node_id] for node_id in bg_node_ids}

print("Connectivity shape:", connectivity.weights.shape)
print("BG node id -> label mapping:")
pprint(bg_mapping)


## Build the TVB Simulator

We construct the TVB side explicitly with `SimulatorBuilder`.

Key developer knobs here are:

- `model`: which TVB mean-field model to use;
- `dt`: TVB integration step;
- `delays_flag`: whether to respect tract delays;
- `connectivity`: the structural connectome handed to TVB;
- model parameters passed to `build(...)`.

For the BG co-simulation in this branch, the interface is specialized for `ReducedWongWangExcIO`.

In [ ]:
output_base = NOTEBOOK_DIR / "output" / "tvb_multiscale_bg_nest_walkthrough"

config = Config(output_base=str(output_base), separate_by_run=False)
config.DEFAULT_NEST_KERNEL_CONFIG["print_time"] = False
config.NEST_VERBOCITY = 30

# The historical code path used NEST 2-style RNG keys.
# Adapt them to NEST 3 so the notebook can run on the current server install.
config.DEFAULT_NEST_KERNEL_CONFIG.pop("grng_seed", None)
config.DEFAULT_NEST_KERNEL_CONFIG.pop("rng_seeds", None)
config.DEFAULT_NEST_KERNEL_CONFIG.setdefault("rng_seed", config.MASTER_SEED + 1)
config.DEFAULT_NEST_KERNEL_CONFIG.setdefault(
    "local_num_threads", config.DEFAULT_NEST_TOTAL_NUM_VIRTUAL_PROCS
)

simulator_builder = SimulatorBuilder()
simulator_builder.config = config
simulator_builder.model = ReducedWongWangExcIO
simulator_builder.connectivity = connectivity
simulator_builder.delays_flag = True
simulator_builder.use_numba = False
simulator_builder.dt = 0.1
simulator_builder.monitor_period = 1.0

# ReducedWongWangExcIO exposes the global coupling parameter G.
simulator = simulator_builder.build(G=np.array([20.0]))

print("TVB model:", type(simulator.model).__name__)
print("TVB dt:", simulator.integrator.dt)
print("TVB regions:", simulator.connectivity.number_of_regions)
print("TVB variables of interest:", simulator.model.variables_of_interest)


## Build the Basal Ganglia Spiking Network in NEST

`BasalGangliaIzhikevichBuilder` is the core SNN builder for the basal ganglia model.

Important implementation details:

- it uses the custom NEST neuron model `izhikevich_hamker`;
- the model source is stored in `tvb_multiscale/tvb_nest/nest/modules/izhikevich_hamker`;
- when the model is not available in the active NEST install, the builder will try to **compile and install it automatically** during the build step;
- the node ids `0..9` are interpreted as:
  `GPe_L, GPe_R, GPi_L, GPi_R, STN_L, STN_R, Striatum_L, Striatum_R, Thal_L, Thal_R`.

For a tutorial notebook we reduce the population order to keep the build lighter than the historical example (`200`).

In [ ]:
bg_builder = BasalGangliaIzhikevichBuilder(simulator, bg_node_ids, config=config)

# Tutorial-friendly size. The repository example uses 200.
bg_builder.population_order = 50

print("Population order:", bg_builder.population_order)
print("Population labels:", [population["label"] for population in bg_builder.populations])
print("\nNode groups used internally by the builder:")
print("  inhibitory (GPe/GPi):", bg_builder.I_nodes)
print("  excitatory (STN/Thalamus):", bg_builder.E_nodes)
print("  striatum:", bg_builder.Istr_nodes_ids)

print("\nInter-node connection templates:")
for connection in bg_builder.nodes_connections:
    print(
        f"- {connection['source']} -> {connection['target']} | "
        f"source_nodes={connection['source_nodes']} | target_nodes={connection['target_nodes']}"
    )


In [ ]:
# This call performs the full NEST-side workflow:
# 1. configure the builder,
# 2. ensure custom NEST models are available,
# 3. create populations,
# 4. connect within-node and across-node projections,
# 5. attach output and input devices.
nest_network = bg_builder.build_spiking_network()

print("Built network type:", type(nest_network).__name__)
print("Built NEST regions:", list(nest_network.brain_regions.index))
print("NEST output device groups:", list(nest_network.output_devices.index))
print("NEST input device groups:", list(nest_network.input_devices.index))
print("NEST min delay:", nest_network.min_delay)


## Build the TVB<->NEST Interface

The BG-specific interface builder is `tvb_multiscale.tvb_nest.interfaces.builders.models.red_ww_basal_ganglia_izhikevich.RedWWexcIOBuilder`.

In this branch it wires the two scales as follows:

- **TVB -> NEST**: the TVB `R` variable is converted to Poisson drive for selected BG populations (`rate` mode);
- **NEST -> TVB**: BG spikes are aggregated and fed back to the TVB `Rin` variable.

This is why the walkthrough uses `ReducedWongWangExcIO` on the TVB side: the interface is specialized for that model contract.

In [ ]:
population_sizes = [int(round(population["scale"] * bg_builder.population_order)) for population in bg_builder.populations]

interface_builder = BasalGangliaRedWWexcIOBuilder(
    simulator,
    nest_network,
    bg_node_ids,
    exclusive_nodes=True,
    populations_sizes=population_sizes,
)

tvb_nest_interface = interface_builder.build_interface(
    tvb_to_nest_mode="rate",
    nest_to_tvb=True,
)

print("Interface type:", type(tvb_nest_interface).__name__)
print("\nTVB -> NEST interface entries:")
print(list(tvb_nest_interface.tvb_to_spikeNet_interfaces.index))
print("\nNEST -> TVB interface entries:")
print(list(tvb_nest_interface.spikeNet_to_tvb_interfaces.index))


## Optional: Run a Short Co-Simulation

Set `RUN_SIMULATION = True` to execute a short run.

Notes:

- `simulator.configure(tvb_nest_interface)` is the point where the TVB simulator becomes a co-simulator;
- the extra one-step NEST run mirrors the repository example and helps multimeters flush the last sample;
- `Cleanup()` is convenient when you are done, but skip it if you want to continue reusing the same NEST kernel later in the notebook.

In [ ]:
RUN_SIMULATION = False
SIMULATION_LENGTH = 20.0

if RUN_SIMULATION:
    simulator.configure(tvb_nest_interface)
    results = simulator.run(simulation_length=SIMULATION_LENGTH)

    # Keep the behavior aligned with examples/tvb_nest/example.py.
    simulator.run_spiking_simulator(
        simulator.tvb_spikeNet_interface.nest_instance.GetKernelStatus("resolution")
    )

    simulator.tvb_spikeNet_interface.nest_instance.Cleanup()
    print(f"Simulation finished. Number of monitor outputs: {len(results)}")
else:
    print("Simulation skipped. Set RUN_SIMULATION = True to execute a short co-simulation.")


In [ ]:
def summarise_monitor_results(results_object):
    summary = []
    for monitor_index, monitor_output in enumerate(results_object):
        times, values = monitor_output
        summary.append(
            {
                "monitor_index": monitor_index,
                "times_shape": np.shape(times),
                "values_shape": np.shape(values),
            }
        )
    return summary


if RUN_SIMULATION:
    pprint(summarise_monitor_results(results))
else:
    print("No results to summarise yet.")


## Inspect the Results

The next cells add quick developer-oriented result inspection:

- plot TVB `R` / `Rin` trajectories for the 10 BG nodes used in the co-simulation;
- summarise the NEST ASCII recorder files written under `output_base / "res" / "nest_recordings"`;
- save quick-look figures and tables into `output_base / "summary_assets"`.

Notes:

- the TVB plot requires `RUN_SIMULATION = True` in the current session;
- the NEST recorder summary can also inspect files already written by an earlier run in the same output directory.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

summary_assets_dir = output_base / "summary_assets"
summary_assets_dir.mkdir(parents=True, exist_ok=True)


def plot_bg_tvb_state_results(results_object, simulator, node_ids, variables=("R", "Rin"), save_dir=None):
    if len(results_object) == 0:
        raise ValueError("No monitor outputs available in `results_object`.")

    times, values = results_object[0]
    variables_of_interest = list(simulator.model.variables_of_interest)
    region_labels = simulator.connectivity.region_labels.tolist()
    plot_variables = [var for var in variables if var in variables_of_interest]

    if len(plot_variables) == 0:
        raise ValueError(
            f"None of the requested variables {variables} exist in variables_of_interest={variables_of_interest}."
        )

    fig, axes = plt.subplots(
        len(plot_variables),
        1,
        figsize=(12, 3.4 * len(plot_variables)),
        sharex=True,
    )
    if len(plot_variables) == 1:
        axes = [axes]

    for ax, variable in zip(axes, plot_variables):
        variable_index = variables_of_interest.index(variable)
        for node_id in node_ids:
            ax.plot(
                times,
                values[:, variable_index, node_id, 0],
                linewidth=1.2,
                label=region_labels[node_id],
            )
        ax.set_ylabel(variable)
        ax.grid(alpha=0.25)

    axes[-1].set_xlabel("Time (ms)")
    axes[0].set_title("TVB trajectories for the BG nodes participating in the co-simulation")
    axes[0].legend(ncol=5, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, 1.35))
    fig.tight_layout()

    save_path = None
    if save_dir is not None:
        save_path = save_dir / "bg_tvb_R_Rin_timeseries.png"
        fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()
    return save_path


if RUN_SIMULATION and "results" in globals():
    tvb_quicklook_path = plot_bg_tvb_state_results(
        results,
        simulator,
        bg_node_ids,
        save_dir=summary_assets_dir,
    )
    print(f"Saved TVB quick-look figure to: {tvb_quicklook_path}")
else:
    print(
        "TVB quick-look plotting is skipped because there is no in-memory `results` object yet. "
        "Set RUN_SIMULATION = True and rerun the simulation cell first."
    )


In [ ]:
recordings_dir = output_base / "res" / "nest_recordings"

if not recordings_dir.exists():
    print(f"Recorder directory does not exist yet: {recordings_dir}")
else:
    rows = []
    for path in sorted(recordings_dir.glob("*.dat")):
        with path.open("r", encoding="utf-8") as stream:
            data_lines = [line.strip() for line in stream if line.strip() and not line.startswith("#")]

        header = data_lines[0].split("\t") if data_lines else []
        records = data_lines[1:] if len(data_lines) > 1 else []
        time_index = header.index("time_ms") if "time_ms" in header else None

        rows.append(
            {
                "file": path.name,
                "bytes": path.stat().st_size,
                "columns": ",".join(header),
                "data_rows": len(records),
                "min_time_ms": float(records[0].split("\t")[time_index]) if records and time_index is not None else None,
                "max_time_ms": float(records[-1].split("\t")[time_index]) if records and time_index is not None else None,
            }
        )

    nest_recordings_summary = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
    summary_csv_path = summary_assets_dir / "nest_recordings_summary.csv"
    nest_recordings_summary.to_csv(summary_csv_path, index=False)

    print(f"Recorder files found: {len(nest_recordings_summary)}")
    print(f"Saved recorder summary to: {summary_csv_path}")
    display(nest_recordings_summary.head(12))

    spike_summary = nest_recordings_summary[
        nest_recordings_summary["file"].str.contains("_spikes_")
    ].copy()

    if len(spike_summary) == 0:
        print("No spike recorder files were found in the current output directory.")
    else:
        spike_summary["label"] = spike_summary["file"].str.replace(r"-\d+-0\.dat$", "", regex=True)
        spike_plot_path = summary_assets_dir / "bg_nest_spike_event_counts.png"

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(spike_summary["label"], spike_summary["data_rows"], color="#4C72B0")
        ax.set_title("Spike rows recorded by NEST spike recorders")
        ax.set_ylabel("Recorded spike rows")
        ax.set_xlabel("Recorder label")
        ax.tick_params(axis="x", rotation=60)
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig.savefig(spike_plot_path, dpi=160, bbox_inches="tight")
        plt.show()

        print(f"Saved spike-count figure to: {spike_plot_path}")
        display(spike_summary[["label", "data_rows", "min_time_ms", "max_time_ms"]])


## Where to Modify the Build as a Developer

The most useful extension points are:

- **TVB model / integration settings**: edit the `SimulatorBuilder` section.
- **BG structural embedding into TVB**: edit the connectivity preprocessing cell.
- **SNN populations and internal BG projections**: inspect and modify `BasalGangliaIzhikevichBuilder`.
- **Background stimulation / DBS devices**: modify `set_input_devices()` in the BG builder.
- **What TVB sends to NEST and what NEST returns to TVB**: modify `BasalGangliaRedWWexcIOBuilder`.
- **Population size / runtime cost tradeoff**: change `bg_builder.population_order`.
- **Custom NEST model compilation**: look at `tvb_multiscale/tvb_nest/nest/modules/izhikevich_hamker` and the compile/install logic in `NESTModelBuilder`.

If you later want the ANNarchy equivalent, the repository contains the parallel example under `examples/tvb_annarchy/basal_ganglia_izhikevich.py`, with the same high-level construction pattern.